# How we built the embedding CSVs

This notebook is **not** part of `tutorial.ipynb`. The main tutorial assumes you already have files like `datasets/train_A_emb.csv` and `datasets/test_emb.csv`. If you want to see exactly how those were produced (FairFace-style labels, train/test splits with different racial mixes, and FaceNet embeddings via DeepFace) work through the cells here.

**What you will do here**

1. Obtain the FairFace label table and images.  
2. Build `test.csv`, `train_A.csv`, `train_B.csv`, and `train_C.csv` with intentional train-set imbalance*(A is balanced, B and C increasingly White-heavy).  
3. Run DeepFace with `model_name="Facenet"` to turn each image path into a numeric vector, then save CSVs into `datasets`.

## Step 1: Get FairFace labels and images

The public **FairFace** resource is distributed on Kaggle ([FairFace dataset](https://www.kaggle.com/datasets/aibloy/fairface)). 

Uncomment **only** when you intend to fetch FairFace programmatically (`kagglehub` + Kaggle API); otherwise skip to **Step 2**.

In [ ]:
# Optional: uncomment when Kaggle CLI / API is configured locally.
#
# import kagglehub
# path = kagglehub.dataset_download("aibloy/fairface")
# print("Dataset files downloaded to:", path)

## Step 2: Load FairFace labels

In [ ]:
import pandas as pd

df = pd.read_csv("FairFace/train_labels.csv")
races = df["race"].unique()
print(sorted(races))


<StringArray>
[     'East Asian',          'Indian',           'Black',           'White',
  'Middle Eastern', 'Latino_Hispanic', 'Southeast Asian']
Length: 7, dtype: str


## Step 3: Decide who goes into train vs test (and imbalance A / B / C)

| Split | Role | Design |
|-------|------|--------|
| **Test** | Shared held-out set | **40** images **per race** (280 rows total) |
| **Train A** | “Balanced” | **150** per race|
| **Train B** | “Moderately imbalanced” | **White** x 450 rows; **each other race** x **100** |
| **Train C** | “Highly imbalanced” | **White** x **700**; **each other race** x **50** |

We sample **without replacement** so that there aren't any duplicated images in any of the dataset.

In [3]:
# testing dataset: 40 per race
test_parts = []
remaining_df = df.copy()

for race in races:
    race_df = remaining_df[remaining_df['race'] == race]
    sample = race_df.sample(n=40, random_state=42)
    test_parts.append(sample)
    remaining_df = remaining_df.drop(sample.index)

test_df = pd.concat(test_parts)

# dataset A: Balanced (150 per race)
A_parts = []

for race in races:
    race_df = remaining_df[remaining_df['race'] == race]
    sample = race_df.sample(n=150, random_state=42)
    A_parts.append(sample)
    remaining_df = remaining_df.drop(sample.index)

A_df = pd.concat(A_parts)

# dataset B: Moderately Imbalanced
dominant_race = 'White'
B_parts = []

# dominant: 450
dom_df = remaining_df[remaining_df['race'] == dominant_race]
dom_sample = dom_df.sample(n=450, random_state=42)
B_parts.append(dom_sample)
remaining_df = remaining_df.drop(dom_sample.index)

# others: ~100 each
for race in races:
    if race == dominant_race:
        continue
    race_df = remaining_df[remaining_df['race'] == race]
    sample = race_df.sample(n=100, random_state=42)
    B_parts.append(sample)
    remaining_df = remaining_df.drop(sample.index)

B_df = pd.concat(B_parts)

# dataset C: Highly Imbalanced
C_parts = []

# dominant: 700
dom_df = remaining_df[remaining_df['race'] == dominant_race]
dom_sample = dom_df.sample(n=700, random_state=42)
C_parts.append(dom_sample)
remaining_df = remaining_df.drop(dom_sample.index)

# others: ~50 each
for race in races:
    if race == dominant_race:
        continue
    race_df = remaining_df[remaining_df['race'] == race]
    sample = race_df.sample(n=50, random_state=42)
    C_parts.append(sample)
    remaining_df = remaining_df.drop(sample.index)

C_df = pd.concat(C_parts)


print("Test:", "size:", len(test_df), test_df['race'].value_counts())
print("A:", "size:", len(A_df), A_df['race'].value_counts())
print("B:", "size:", len(B_df), B_df['race'].value_counts())
print("C:", "size:", len(C_df), C_df['race'].value_counts())

Test: size: 280 race
East Asian         40
Indian             40
Black              40
White              40
Middle Eastern     40
Latino_Hispanic    40
Southeast Asian    40
Name: count, dtype: int64
A: size: 1050 race
East Asian         150
Indian             150
Black              150
White              150
Middle Eastern     150
Latino_Hispanic    150
Southeast Asian    150
Name: count, dtype: int64
B: size: 1050 race
White              450
East Asian         100
Indian             100
Black              100
Middle Eastern     100
Latino_Hispanic    100
Southeast Asian    100
Name: count, dtype: int64
C: size: 1000 race
White              700
East Asian          50
Indian              50
Black               50
Middle Eastern      50
Latino_Hispanic     50
Southeast Asian     50
Name: count, dtype: int64


## Step 4: Save intermediate path lists (`test.csv`, `train_A/B/C.csv`)

In [ ]:
test_df.to_csv('/FairFace/test.csv', index=False)
A_df.to_csv('/FairFace/train_A.csv', index=False)
B_df.to_csv('/FairFace/train_B.csv', index=False)
C_df.to_csv('/FairFace/train_C.csv', index=False)

## Step 5: FaceNet embeddings with DeepFace

[DeepFace](https://github.com/serengil/deepface) uses a pretrained FaceNet model. For each row, we call `DeepFace.represent` and collect the `embedding` vector.

In [ ]:
from deepface import DeepFace
import numpy as np
import os
import pandas as pd

IMG_ROOT = "../FairFace"
# tutorial.ipynb uses the first 128 columns as features 
EXPECTED_EMBEDDING_DIM = 128


def make_embeddings(csv_in: str, csv_out: str) -> None:
    """Read paths from csv_in, append FaceNet embedding columns, write csv_out."""

    frame = pd.read_csv(csv_in)
    embeddings = []

    for relative_path in frame["file"]:
        full_path = os.path.join(IMG_ROOT, relative_path)

        if not os.path.exists(full_path):
            print("Missing:", full_path)
            embeddings.append([np.nan] * EXPECTED_EMBEDDING_DIM)
            continue

        vec = DeepFace.represent(
            img_path=full_path,
            model_name="Facenet",
            enforce_detection=False,
            detector_backend="skip",
        )[0]["embedding"]

        if len(vec) != EXPECTED_EMBEDDING_DIM:
            raise ValueError(
                f"Embedding dim {len(vec)} != {EXPECTED_EMBEDDING_DIM}. "
                "Update EXPECTED_EMBEDDING_DIM and tutorial feature_cols, or change model_name."
            )

        embeddings.append(vec)

    emb_frame = pd.DataFrame(embeddings)
    out = pd.concat([emb_frame, frame.reset_index(drop=True)], axis=1)
    out.to_csv(csv_out, index=False)
    print(f"Wrote {len(out)} rows -> {csv_out}")


make_embeddings("/FairFace/train_A.csv", "../datasets/train_A_emb.csv")
make_embeddings("/FairFace/train_B.csv", "../datasets/train_B_emb.csv")
make_embeddings("/FairFace/train_C.csv", "../datasets/train_C_emb.csv")
make_embeddings("/FairFace/test.csv", "../datasets/test_emb.csv")